# 01 - Data cleaning: InhA IC50 data from ChEMBL

**Goal:** turn the raw ChEMBL download for InhA (target CHEMBL1849) into one clean pIC50 value per molecule.

**Input:** the raw ChEMBL zip. Upload it to the Files panel; it is found automatically, whatever its file name.
**Outputs:** `inha_ic50_final.csv` (342 molecules, before duplicate merge) and `inha_ic50_final_341.csv` (341 molecules, with scaffold column).

Steps: filter to IC50 / '=' / nM, look at why raw nM values cannot simply be averaged, remove records ChEMBL flagged as outside the typical range, convert to pIC50 first and then take the median per molecule, merge one salt duplicate, add Bemis-Murcko scaffolds.

In [ ]:
# Install RDKit only if it is missing (needed on Colab; already present locally)
try:
    import rdkit
except ImportError:
    %pip install -q rdkit==2026.3.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 59.2 MB/s eta 0:00:00


## 1. Load the raw download and filter

In [ ]:
import pandas as pd, numpy as np

import glob
zips = glob.glob("*.zip")     # finds the raw zip whatever it is called (also names like x.zip.zip)
print("Zip files found:", zips)
if len(zips) != 1:
    raise FileNotFoundError(f"Expected exactly one .zip file in this folder, found {len(zips)}: {zips}")
RAW_ZIP = zips[0]

# pandas can read the zip directly, no need to unzip
df = pd.read_csv(RAW_ZIP, sep=";", low_memory=False)

# Standard Relation values contain the quotes themselves, like '=' so strip them
df["Standard Relation"] = df["Standard Relation"].astype(str).str.strip("'\" ")

print("1. Rows, Columns:", df.shape)
print("\n2. Counts per Standard Type:")
print(df["Standard Type"].value_counts(dropna=False))

ic50_eq = df[(df["Standard Type"] == "IC50") & (df["Standard Relation"] == "=")]
print("\n3. IC50 '=' rows:", len(ic50_eq))
print("4. Unique molecules (SMILES):", ic50_eq["Smiles"].dropna().nunique())
print("   Unique molecules (ChEMBL ID):", ic50_eq["Molecule ChEMBL ID"].dropna().nunique())

Zip files found: ['chembl_CHEMBL1849_bioactivities_raw.zip']
1. Rows, Columns: (1460, 48)

2. Counts per Standard Type:
Standard Type
Inhibition    605
IC50          488
MIC           122
Ki             86
Activity       35
Kd             29
MIC99          21
Kii            15
TDI            12
TIME           10
Kis             9
Delta Tm        8
deltaG          5
deltaH          5
deltaS          5
Tm              4
Ratio Ki        1
Name: count, dtype: int64

3. IC50 '=' rows: 403
4. Unique molecules (SMILES): 346
   Unique molecules (ChEMBL ID): 346


## 2. Why raw nM values must not be averaged

An earlier version took the median of raw nM values per molecule. For a molecule with two records, that is just their average, and one huge value dominates. Two examples:

In [ ]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 250)

cols = ["Molecule ChEMBL ID", "Standard Value", "Standard Units",
        "Assay ChEMBL ID", "Assay Description", "Data Validity Comment"]

for mol in ["CHEMBL260399", "CHEMBL264682"]:
    print(mol)
    display(ic50_eq[ic50_eq["Molecule ChEMBL ID"] == mol][cols])
    print()

CHEMBL260399


,Molecule ChEMBL ID,Standard Value,Standard Units,Assay ChEMBL ID,Assay Description,Data Validity Comment
4,CHEMBL260399,130.00,nM,CHEMBL936672,Inhibition of 100 nM Mycobacterium tuberculosis H37RV InhA,NaN
504,CHEMBL260399,7709034.69,nM,CHEMBL3047415,Inhibition of Mycobacterium tuberculosis InhA,Outside typical range



CHEMBL264682


,Molecule ChEMBL ID,Standard Value,Standard Units,Assay ChEMBL ID,Assay Description,Data Validity Comment
95,CHEMBL264682,90928494.39,nM,CHEMBL3047415,Inhibition of Mycobacterium tuberculosis InhA,Outside typical range
361,CHEMBL264682,11.00,nM,CHEMBL936673,Inhibition of 1 nM Mycobacterium tuberculosis H37RV InhA,NaN


Each molecule has one normal value (130 nM, 11 nM) and one value in the millions that ChEMBL itself marked "Outside typical range". Averaging the two gives a meaningless number.

## 3. Which assays do the flagged records come from?

In [ ]:
d = ic50_eq.copy()
d["IC50_nM"] = pd.to_numeric(d["Standard Value"], errors="coerce")
d = d[(d["Standard Units"] == "nM") & (d["IC50_nM"] > 0)]

print("IC50 '=' rows in nM:", len(d))
print(d[d["Data Validity Comment"].notna()]["Assay ChEMBL ID"].value_counts())

IC50 '=' rows in nM: 402
Assay ChEMBL ID
CHEMBL3047415    20
CHEMBL4380941     2
CHEMBL5623729     1
Name: count, dtype: int64


Check of the removed values (used in the README): the range of the flagged IC50 values, the flagged records from the two smaller assays, and the unflagged records from the suspicious assay that stay in the data.

In [ ]:
dd = ic50_eq.copy()
dd["IC50_nM"] = pd.to_numeric(dd["Standard Value"], errors="coerce")
dd = dd[(dd["Standard Units"] == "nM") & (dd["IC50_nM"] > 0)]
fl = dd[dd["Data Validity Comment"].notna()]
print("Flagged IC50 range (nM):", round(fl["IC50_nM"].min()), "to", round(fl["IC50_nM"].max()))
print("\nFlagged records from the two smaller assays:")
print(fl[fl["Assay ChEMBL ID"] != "CHEMBL3047415"][["Molecule ChEMBL ID", "IC50_nM", "Assay ChEMBL ID"]].to_string(index=False))
print("\nUnflagged records from assay CHEMBL3047415:")
x = dd[(dd["Assay ChEMBL ID"] == "CHEMBL3047415") & (dd["Data Validity Comment"].isna())]
print(x[["Molecule ChEMBL ID", "IC50_nM"]].to_string(index=False))


Flagged IC50 range (nM): 122180 to 90928494

Flagged records from the two smaller assays:
Molecule ChEMBL ID  IC50_nM Assay ChEMBL ID
     CHEMBL4454539 127000.0   CHEMBL4380941
     CHEMBL4797728 280000.0   CHEMBL5623729
     CHEMBL4557187 291000.0   CHEMBL4380941

Unflagged records from assay CHEMBL3047415:
Molecule ChEMBL ID  IC50_nM
      CHEMBL264679 87096.36


20 of the 23 flagged records come from assay CHEMBL3047415, the other 3 from two other assays. Removing all flagged records is a **mechanical rule**, not a judgement on each value.

## 4. Clean: drop flagged records, pIC50 first, then median per molecule

In [ ]:
import numpy as np, pandas as pd

# Start from ic50_eq (IC50 rows with '=' relation, from section 1)
d = ic50_eq.copy()
d["IC50_nM"] = pd.to_numeric(d["Standard Value"], errors="coerce")
d = d[(d["Standard Units"] == "nM") & (d["IC50_nM"] > 0)]
print("IC50 '=' rows in nM:", len(d), "| molecules:", d["Molecule ChEMBL ID"].nunique())

# 1) Drop records ChEMBL itself flagged as suspicious
flagged = d["Data Validity Comment"].notna()
print("Flagged rows removed:", flagged.sum())
print(d.loc[flagged, "Data Validity Comment"].value_counts())
d = d[~flagged].copy()

# 2) Convert each record to pIC50 FIRST (log scale)
d["pIC50"] = 9 - np.log10(d["IC50_nM"])

# 3) Then take the median per molecule
g = d.groupby(["Molecule ChEMBL ID", "Smiles"])["pIC50"]
final = g.agg(pIC50="median", n_records="count", spread=lambda s: s.max() - s.min()).reset_index()
final["IC50_nM"] = 10 ** (9 - final["pIC50"])

print("\nFinal molecules:", len(final))
print("Molecules with >1 record:", (final["n_records"] > 1).sum())
print("Molecules whose records differ by >1 log unit:", (final["spread"] > 1).sum())
print("Molecules whose records differ by >2 log units:", (final["spread"] > 2).sum())
print(final["pIC50"].describe())
print("Largest IC50_nM now:", round(final["IC50_nM"].max(), 1))

final.to_csv("inha_ic50_final.csv", index=False)
print("\nSaved: inha_ic50_final.csv")

IC50 '=' rows in nM: 402 | molecules: 345
Flagged rows removed: 23
Data Validity Comment
Outside typical range    23
Name: count, dtype: int64

Final molecules: 342
Molecules with >1 record: 23
Molecules whose records differ by >1 log unit: 5
Molecules whose records differ by >2 log units: 0
count    342.000000
mean       5.889795
std        1.180192
min        3.400000
25%        4.980053
50%        5.537602
75%        6.683075
max        8.698970
Name: pIC50, dtype: float64
Largest IC50_nM now: 398107.2

Saved: inha_ic50_final.csv


## 5. Duplicates and scaffolds

Keep only the largest fragment (some entries are salts with a `.` in the SMILES), write a canonical SMILES, and count molecules that become identical.

In [ ]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv("inha_ic50_final.csv")
chooser = rdMolStandardize.LargestFragmentChooser()

def largest_fragment(smi):
    return chooser.choose(Chem.MolFromSmiles(smi))

# --- Task 1: duplicates ---
print("Entries with '.' in SMILES (salts/mixtures):", df["Smiles"].str.contains(r"\.", regex=True).sum())
df["canon"] = df["Smiles"].apply(lambda s: Chem.MolToSmiles(largest_fragment(s)))
n_unique = df["canon"].nunique()
print("Molecules:", len(df), "| unique canonical SMILES:", n_unique, "| duplicates:", len(df) - n_unique)

dup = df[df.duplicated("canon", keep=False)].sort_values("canon")
print("\nIdentical pairs/groups:")
print(dup[["Molecule ChEMBL ID", "Smiles", "canon", "pIC50"]].to_string())

# --- Task 2: Bemis-Murcko scaffolds ---
df["scaffold"] = df["canon"].apply(lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s))
print("\nUnique scaffolds:", df["scaffold"].nunique())
print("Molecules with empty scaffold (no ring):", (df["scaffold"] == "").sum())
print("\nFive largest scaffold groups:")
print(df["scaffold"].value_counts().head(5))
print("Scaffolds that appear only once:", (df["scaffold"].value_counts() == 1).sum())


Entries with '.' in SMILES (salts/mixtures): 1
Molecules: 342 | unique canonical SMILES: 341 | duplicates: 1

Identical pairs/groups:
    Molecule ChEMBL ID                                                              Smiles                                                         canon     pIC50
11        CHEMBL216689        O=C(Nc1cc(-c2ccccc2)c(O)c(-c2ccccc2)c1)C1CC(=O)N(C2CCCCC2)C1  O=C(Nc1cc(-c2ccccc2)c(O)c(-c2ccccc2)c1)C1CC(=O)N(C2CCCCC2)C1  6.853872
261       CHEMBL539344  Cl.Cl.O=C(Nc1cc(-c2ccccc2)c(O)c(-c2ccccc2)c1)C1CC(=O)N(C2CCCCC2)C1  O=C(Nc1cc(-c2ccccc2)c(O)c(-c2ccccc2)c1)C1CC(=O)N(C2CCCCC2)C1  7.207608

Unique scaffolds: 144
Molecules with empty scaffold (no ring): 0

Five largest scaffold groups:
scaffold
O=C(Nc1ccccc1)C1CC(=O)N(C2CCCCC2)C1            26
O=C(CN1C(=O)C(=Cc2ccccc2)SC1=S)c1ccccc1        23
c1ccc(Oc2ccccc2)cc1                            20
O=C(NC1CCN(C(=O)c2coc3ccccc23)C1)c1ccn[nH]1    19
c1ccc2c(Nc3ccc(N4CCCCC4)cc3)ccnc2c1            16
Name: count, dtype: i

## 6. Merge the duplicate and save the 341-molecule file

CHEMBL216689 and CHEMBL539344 are the same molecule (the second is a dihydrochloride salt). They are merged by averaging their pIC50 (6.854 and 7.208 -> 7.031), keeping CHEMBL216689.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv("inha_ic50_final.csv")          # the 342-molecule file stays untouched
chooser = rdMolStandardize.LargestFragmentChooser()
df["canon"] = df["Smiles"].apply(lambda s: Chem.MolToSmiles(chooser.choose(Chem.MolFromSmiles(s))))

# one row per canonical structure: average pIC50 of the duplicates,
# keep the ID of the plain (non-salt) form
df["is_salt"] = df["Smiles"].str.contains(r"\.", regex=True)
df = df.sort_values(["canon", "is_salt"])
merged = df.groupby("canon", as_index=False).agg(
    **{"Molecule ChEMBL ID": ("Molecule ChEMBL ID", "first"),
       "pIC50": ("pIC50", "mean")}
)
merged["Smiles"] = merged["canon"]
merged["IC50_nM"] = 10 ** (9 - merged["pIC50"])
merged["scaffold"] = merged["canon"].apply(lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s))
merged = merged[["Molecule ChEMBL ID", "Smiles", "pIC50", "IC50_nM", "scaffold"]].reset_index(drop=True)

print("Molecules after merge:", len(merged))
print(merged[merged["Molecule ChEMBL ID"] == "CHEMBL216689"][["Molecule ChEMBL ID", "pIC50"]])
merged.to_csv("inha_ic50_final_341.csv", index=False)

Molecules after merge: 341
    Molecule ChEMBL ID    pIC50
269       CHEMBL216689  7.03074
